# 04. 탐색적 데이터 분석 (EDA)

**목표**: 반려견 질병 말뭉치의 분포와 패턴을 시각화하여 중간 인사이트 도출

| 분석 항목 | 내용 |
|----------|------|
| 생애주기별 질병 빈도 | 자견/성견/노령견별 주요 질병 분포 |
| 진료과별 분포 | 5개 진료과 데이터 현황 |
| 워드클라우드 | 생애주기별 핵심 키워드 시각화 |
| 텍스트 길이 분포 | Q&A 길이 통계 |
| 질병-진료과 교차 분석 | 히트맵 |

> **보고서 연계**: 4번(EDA), 5번(중간 인사이트) 항목 직결

## 0. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
from collections import Counter

from utils.config import DATA_PROCESSED

# 한글 폰트
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

PREPROCESSED_PATH = DATA_PROCESSED / 'corpus_preprocessed.csv'
EDA_OUTPUT        = DATA_PROCESSED / 'eda_figures'
EDA_OUTPUT.mkdir(exist_ok=True)

print("환경 설정 완료")

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(PREPROCESSED_PATH)
print(f"전체 레코드: {len(df):,}개")
print(f"컬럼: {list(df.columns)}")
df.head(2)

## 2. 기초 통계

In [ ]:
print("=== 생애주기 분포 ===")
lc = df['lifeCycle'].value_counts()
for k, v in lc.items():
    print(f"  {k:6s}: {v:,}개 ({v/len(df)*100:.1f}%)")

print("\n=== 진료과 분포 ===")
dept = df['department'].value_counts()
for k, v in dept.items():
    print(f"  {k:5s}: {v:,}개 ({v/len(df)*100:.1f}%)")

print(f"\n=== 고유 질병 수 ===")
print(f"  {df['disease'].nunique():,}종")

## 3. 생애주기별 질병 빈도 (파이차트)

In [ ]:
lifecycle_order = ['자견', '성견', '노령견']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lc_name in zip(axes, lifecycle_order):
    subset = df[df['lifeCycle'] == lc_name]['disease'].value_counts().head(8)
    ax.pie(
        subset.values,
        labels=subset.index,
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': 9}
    )
    ax.set_title(f'{lc_name} 주요 질병 Top 8', fontsize=13, fontweight='bold')

plt.suptitle('생애주기별 주요 질병 분포', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(EDA_OUTPUT / 'lifecycle_disease_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 생애주기 × 진료과 교차 분석 (히트맵)

In [ ]:
crosstab = pd.crosstab(df['lifeCycle'], df['department'])
crosstab = crosstab.reindex(lifecycle_order)  # 순서 고정

fig = px.imshow(
    crosstab,
    text_auto=True,
    color_continuous_scale='Blues',
    title='생애주기 × 진료과 교차 분석',
    labels=dict(x='진료과', y='생애주기', color='레코드 수')
)
fig.update_layout(font=dict(size=13))
fig.write_html(str(EDA_OUTPUT / 'lifecycle_dept_heatmap.html'))
fig.show()

## 5. 생애주기별 워드클라우드

In [ ]:
# 한글 폰트 경로 (macOS 기준)
FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, lc_name in zip(axes, lifecycle_order):
    tokens = ' '.join(
        df[df['lifeCycle'] == lc_name]['input_tokens'].dropna()
    )
    wc = WordCloud(
        font_path=FONT_PATH,
        width=600, height=400,
        background_color='white',
        max_words=60,
        colormap='Blues'
    ).generate(tokens)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{lc_name} 키워드', fontsize=13, fontweight='bold')

plt.suptitle('생애주기별 질문 키워드 워드클라우드', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_OUTPUT / 'wordcloud_lifecycle.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 진료과별 Top 10 질병 (막대 그래프)

In [ ]:
departments = df['department'].unique()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, dept_name in zip(axes, departments):
    top10 = df[df['department'] == dept_name]['disease'].value_counts().head(10)
    ax.barh(top10.index[::-1], top10.values[::-1], color='#4C72B0')
    ax.set_title(f'{dept_name} Top 10 질병', fontweight='bold')
    ax.set_xlabel('빈도')
    for i, v in enumerate(top10.values[::-1]):
        ax.text(v + 1, i, str(v), va='center', fontsize=8)

axes[-1].set_visible(False)  # 빈 subplot 숨김
plt.suptitle('진료과별 Top 10 질병', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_OUTPUT / 'dept_top10_disease.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Q&A 텍스트 길이 분포

In [ ]:
df['input_len']  = df['input_clean'].str.len()
df['output_len'] = df['output_clean'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['input_len'],  bins=50, color='#4C72B0', edgecolor='white')
axes[0].set_title('질문(input) 길이 분포')
axes[0].set_xlabel('글자 수')
axes[0].axvline(df['input_len'].median(), color='red', linestyle='--', label=f'중앙값: {df["input_len"].median():.0f}')
axes[0].legend()

axes[1].hist(df['output_len'], bins=50, color='#DD8452', edgecolor='white')
axes[1].set_title('답변(output) 길이 분포')
axes[1].set_xlabel('글자 수')
axes[1].axvline(df['output_len'].median(), color='red', linestyle='--', label=f'중앙값: {df["output_len"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(EDA_OUTPUT / 'text_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(df[['input_len', 'output_len']].describe().round(1))

## 8. 중간 인사이트 (보고서 5번 항목)

In [ ]:
print("=== 생애주기별 주요 질병 Top 3 ===")
for lc_name in lifecycle_order:
    top3 = df[df['lifeCycle'] == lc_name]['disease'].value_counts().head(3)
    print(f"\n[{lc_name}]")
    for disease, count in top3.items():
        print(f"  {disease}: {count:,}건")

## 9. EDA 요약

| 항목 | 결과 |
|------|------|
| 전체 레코드 | (실행 후 기입) |
| 고유 질병 수 | (실행 후 기입) |
| 자견 주요 질병 | (실행 후 기입) |
| 성견 주요 질병 | (실행 후 기입) |
| 노령견 주요 질병 | (실행 후 기입) |
| 생성된 시각화 | lifecycle_disease_pie.png, wordcloud_lifecycle.png, dept_top10_disease.png, lifecycle_dept_heatmap.html |

### 중간 인사이트
- (실행 후 발견된 패턴 기입)
- (예: 노령견에서 종양 관련 질병 빈도가 높음)
- (예: 피부과 질환은 성견에 집중)

> **다음 단계**: `05_ground_truth.ipynb` — 평가셋 수동 구축 (MRR/Top-K 측정 기준)